# PECT-JEPA: Pipeline Huấn Luyện & Đánh Giá Đa Tác Vụ Chuẩn (Master Reference Pipeline)
## Unified 64D Latent + Dodd-Deeds Spectral Phase AdaLN + Test-Time Adaptation (TTA)

**Mục đích file:** Đây là notebook chuẩn mẫu (Master Reference Pipeline). Đối với từng thử nghiệm cụ thể (EXP-70, EXP-71,...), vui lòng sao chép hoặc mở file notebook riêng tương ứng (ví dụ: `train_70_full_cross_pairing.ipynb`).

**Mục tiêu nghiên cứu & đồ án tốt nghiệp:**
1. **Self-Supervised Pretraining (JEPA)**:
   - Huấn luyện Self-Supervised trên toàn bộ 81 file C-scans (2,106 cặp điều kiện định hướng A->B != B->A trên 3 mẫu tấm Corrosion, Rivet_v1, Rivet_v2) với spatial grid stride = 4 (nửa bán kính r3=7) để tối ưu thời gian (~3.5 phút/epoch) và phủ 100% không gian đo.
   - Downstream Evaluation toàn diện với stride = 1 (toàn bộ 72,900 điểm ảnh) cho các tác vụ Anomaly Detection, Flaw Sizing, Segmentation, và TTA.
   - Unified 64D latent (bảo toàn full-rank dual-subspace không phân mảnh).
   - Dodd-Deeds 16D spectral phase condition token với điều biến AdaLN.
   - Normalized L1 prediction loss loại bỏ hiện tượng magnitude-drift.
   - Reference patch trung vị 4 góc (`corner_median`) đảm bảo 0% khuyết tật rò rỉ.
2. **Downstream Probe Fitting (Frozen Features)**:
   - Huấn luyện Dual Probes: Raw Baseline Probes & TTA Calibrated Probes (sử dụng 16 scan Base Domain Train + Val).
3. **Comprehensive Downstream Multi-Task Evaluation**:
   - **Task 1: Anomaly Detection**: Xuất ảnh 4 subplots (CAD Label, Raw Peak-to-Peak $V_{\text{pp}}$, Unsupervised Mahalanobis, Linear Probe defect probability).
   - **Task 2: Flaw Sizing Hurdle Protocol**: $R^2$, MAE (mm), RMSE (mm) trên mẫu Corrosion kèm biểu đồ tán xạ hiệu chuẩn.
   - **Task 3: Defect Segmentation**: Ngưỡng hóa xác suất, tính IoU (Jaccard), Dice F1, Precision, Recall.
   - **Task 4: Canonical Test-Time Adaptation (TTA) Benchmark**: Xuất ảnh 5 subplots so sánh (CAD Label, Raw $V_{\text{pp}}$, Zero-Shot LP, TTA Calibrated LP, TTA Mahalanobis Energy) trên các ca dịch chuyển sensor, waveform, liftoff.
   - **Readout Diagnostics**: Đo lường định lượng các cơ chế tự hiệu chuẩn P0, P1, P1u_s2 (zero-GT).


In [ ]:
from google.colab import drive
import os, glob

drive.mount('/content/drive', force_remount=True)

DATA = '/content/drive/MyDrive/PECT_Dataset/'
DRIVE_RUNS = '/content/drive/MyDrive/PECT_Dataset/runs/'
os.makedirs(DRIVE_RUNS, exist_ok=True)

tdms_files = [f for f in glob.glob(os.path.join(DATA, '**', '*.tdms'), recursive=True) if not f.endswith('_index.tdms')]
print(f'[*] Tìm thấy {len(tdms_files)}/81 files TDMS')

for p in [
    os.path.join(DATA, 'ground_truth/corrosion/corrosion_gt_mask.npy'),
    os.path.join(DATA, 'ground_truth/rivet_v1/rivet_v1_gt_mask.npy'),
    os.path.join(DATA, 'ground_truth/rivet_v2/rivet_v2_gt_mask.npy')
]:
    status = 'OK  ' if os.path.exists(p) else 'MISS'
    print(f'[{status}] {p}')


In [ ]:
# Đồng bộ mã nguồn mới nhất từ remote branch Define_PECT_JEPA
!test -d GraduationProject && (cd GraduationProject && git fetch origin && git checkout Define_PECT_JEPA && git reset --hard origin/Define_PECT_JEPA) || git clone -b Define_PECT_JEPA https://github.com/TranAnh35/GraduationProject.git
!cd GraduationProject && git log --oneline -3


In [ ]:
# Cài đặt môi trường chuẩn khớp với phòng thí nghiệm
!pip install -q -U "scikit-learn==1.7.2" nptdms scipy tqdm matplotlib joblib pandas 2>&1 | tail -3

import torch, sklearn, scipy, nptdms
print('=' * 60)
print(f'PyTorch:      {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:         {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
print(f'Scikit-Learn: {sklearn.__version__}')
print('=' * 60)


In [ ]:
# Symlink thư mục runs nội bộ sang Google Drive để lưu checkpoints và kết quả an toàn
!cd GraduationProject && mkdir -p experiments/cross_condition && if [ ! -L experiments/cross_condition/runs ]; then mv experiments/cross_condition/runs/* "$DRIVE_RUNS/" 2>/dev/null; rm -rf experiments/cross_condition/runs && ln -s "$DRIVE_RUNS" experiments/cross_condition/runs && echo '[OK] LINKED'; else echo '[OK] ALREADY_LINKED'; fi
!ln -sfn "$DRIVE_RUNS" GraduationProject/experiments/cross_condition/runs
!ls -la GraduationProject/experiments/cross_condition/ | grep runs


In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'


# ===== CẤU HÌNH THỰC NGHIỆM EXP-70 (Full 81-File Cross-Condition Pairing + Grid Stride 4) =====
EXP_NAME = 'exp70_full_cross_pairing'

BASE_FLAGS = '--star_radii 2 4 7 --tokenizer_type energy_adaptive --latent_disentanglement false --unified_latent true --two_stream_backbone false --embed_dim 64 --var_weight 0.5 --cov_weight 0.2 --var_gamma 1.0 --scan_centered_vicreg true --inv_start_epoch 11 --inv_ramp_epochs 20 --inv_cosine true --scan_centered_inv true --inv_weight 0.2 --diverge_patience 0 --device cuda --use_full_dataset true --grid_stride 4'
EXTRA_FLAGS = '--continuous_pos_embed true --contiguous_masking true --relative_energy_anchor true --bi_directional_relative_energy true --patch_polarity_align true --spectral_condition_token true --spectral_phase_encoding cos_sin --use_condition_token true --condition_modulation adaln --pred_loss_type normalized_l1 --anti_hijack_ratio 0.0 --self_focus_gamma 0.0 --ref_patch_mode corner_median'

os.environ.update({
    'EXP_NAME': EXP_NAME,
    'BASE_FLAGS': BASE_FLAGS,
    'EXTRA_FLAGS': EXTRA_FLAGS,
    'DATA': DATA,
    'DRIVE_RUNS': DRIVE_RUNS
})

print(f'[*] Thí nghiệm: {EXP_NAME}')
print(f'[*] Flags: {BASE_FLAGS} {EXTRA_FLAGS}')


In [ ]:
# Huấn luyện mô hình PECT-JEPA từ đầu (100 epochs, eval val mỗi epoch)
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train \
    --exp_name "$EXP_NAME" \
    --add_timestamp true \
    $BASE_FLAGS \
    $EXTRA_FLAGS \
    --data_dir "$DATA" \
    --epochs 100 \
    --batch_size 256 \
    --eval_val_every 1


In [ ]:
# [TÙY CHỌN] Chạy cell này CHỈ KHI muốn tiếp tục quá trình train bị gián đoạn từ Google Drive
import os, glob

candidates = sorted(glob.glob(os.path.join(DRIVE_RUNS, f"{EXP_NAME}_*")), key=os.path.getmtime)
if not candidates:
    raise FileNotFoundError(f'Không tìm thấy thư mục run nào cho {EXP_NAME} trong {DRIVE_RUNS}')
run_dir = candidates[-1]

ckpt_latest = os.path.join(run_dir, 'checkpoints', 'latest_cross_condition_jepa.pt')
ckpt_best = os.path.join(run_dir, 'checkpoints', 'best_cross_condition_jepa.pt')
resume_ckpt = ckpt_latest if os.path.exists(ckpt_latest) else ckpt_best

print(f'[*] Resume từ checkpoint: {resume_ckpt}')
os.environ['RESUME_CKPT'] = resume_ckpt

!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train \
    --exp_name "$EXP_NAME" \
    $BASE_FLAGS \
    $EXTRA_FLAGS \
    --data_dir "$DATA" \
    --epochs 100 \
    --batch_size 256 \
    --eval_val_every 1 \
    --resume_checkpoint "$RESUME_CKPT"


In [ ]:
# ==============================================================================
# Huấn luyện probe downstream (Linear/MLP Detection & Ridge/MLP Sizing)
# Bộ mô hình bao gồm: Raw Probes (Baseline) và TTA Whitened Probes (Test-Time Adaptation)
# ==============================================================================
import os, glob

candidates = sorted(glob.glob(os.path.join(DRIVE_RUNS, f"{EXP_NAME}_*")), key=os.path.getmtime)
if not candidates:
    raise FileNotFoundError(f'Không tìm thấy run folder cho {EXP_NAME} trong {DRIVE_RUNS}')
run_dir = candidates[-1]

ckpt_best = os.path.join(run_dir, 'checkpoints', 'best_cross_condition_jepa.pt')
ckpt_latest = os.path.join(run_dir, 'checkpoints', 'latest_cross_condition_jepa.pt')
eval_ckpt = ckpt_best if os.path.exists(ckpt_best) else ckpt_latest

downstream_dir = os.path.join(run_dir, 'downstream_models')
os.makedirs(downstream_dir, exist_ok=True)

os.environ['EVAL_CKPT'] = eval_ckpt
os.environ['DOWNSTREAM_DIR'] = downstream_dir
os.environ['RUN_DIR'] = run_dir

print(f'[*] Checkpoint đánh giá: {eval_ckpt}')
print(f'[*] Thư mục lưu probe:  {downstream_dir}')

!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.downstream.train_downstream \
    --checkpoint "$EVAL_CKPT" \
    --data_dir "$DATA" \
    --out_dir "$DOWNSTREAM_DIR" \
    --device cuda \
    --tta_trim_ratio 0.75 \
    --tta_shrinkage 0.01


In [ ]:
# ==============================================================================
# Đánh giá toàn diện 4 tasks trên toàn bộ 81 scans (12 Train, 4 Val, 65 Test OOD)
# 1. Task 1: Anomaly Detection (xuất ảnh 4 subplots: CAD GT, Raw Vpp, Mahalanobis, Linear Probe)
# 2. Task 2: Flaw Sizing Hurdle Protocol (R^2, MAE mm, RMSE mm kèm scatter plot)
# 3. Task 3: Defect Segmentation (IoU, Dice F1, Precision, Recall)
# 4. Task 4: Canonical Test-Time Adaptation (TTA) Benchmark (xuất ảnh 5 subplots: CAD GT, Raw Vpp, Zero-Shot LP, TTA LP, TTA Mahalanobis)
# ==============================================================================
eval_out_dir = os.path.join(run_dir, 'evaluation_results')
os.environ['EVAL_OUT_DIR'] = eval_out_dir

!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.evaluate \
    --checkpoint "$EVAL_CKPT" \
    --data_dir "$DATA" \
    --downstream_dir "$DOWNSTREAM_DIR" \
    --out_dir "$EVAL_OUT_DIR" \
    --device cuda \
    --eval_tta true \
    --tta_trim_ratio 0.75 \
    --tta_shrinkage 0.01


In [ ]:
# ==============================================================================
# Chẩn đoán sâu hình học không gian tiềm ẩn và các cơ chế hiệu chuẩn per-scan (P0, P1, P1u_s2, Otsu)
# ==============================================================================
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.diagnose_readouts \
    --checkpoint "$EVAL_CKPT" \
    --data_dir "$DATA" \
    --device cuda


In [ ]:
# ==============================================================================
# BẢNG TỔNG HỢP KẾT QUẢ ĐỊNH LƯỢNG (DOWNSTREAM MASTER BENCHMARK & TTA)
# ==============================================================================
import json
import pandas as pd
from IPython.display import display, HTML

summary_path = os.path.join(eval_out_dir, 'master_benchmark_summary.json')
if os.path.exists(summary_path):
    with open(summary_path, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    
    print('=' * 80)
    print(' MASTER BENCHMARK EVALUATION SUMMARY')
    print('=' * 80)
    
    # 1. Bảng theo Splits
    splits_data = []
    for sp, data in summary.get('by_split', {}).items():
        splits_data.append({
            'Split': sp.upper(),
            'Files': data.get('count', 0),
            'LP AUC': f"{data.get('linear_probe_auc_mean', 0.0):.4f} ± {data.get('linear_probe_auc_std', 0.0):.4f}",
            'LP AP': f"{data.get('linear_probe_ap_mean', 0.0):.4f}",
            'Maha AUC': f"{data.get('mahalanobis_auc_mean', 0.0):.4f}",
            'CNR': f"{data.get('cnr_mean', 0.0):.2f}",
            'Seg IoU': f"{data.get('segmentation_iou_mean', 0.0):.4f}",
            'Seg Dice': f"{data.get('segmentation_dice_mean', 0.0):.4f}",
            'Sizing R²': f"{data.get('sizing_r2_mean', 0.0):.4f}",
            'MAE (mm)': f"{data.get('sizing_mae_mean', 0.0):.3f}"
        })
    df_splits = pd.DataFrame(splits_data)
    display(HTML('<h3>1. Hiệu năng theo Tập dữ liệu (Splits)</h3>'))
    display(df_splits)
    
    # 2. Bảng theo Cảm biến (Sensor Shift)
    sensors_data = []
    for sn, data in summary.get('by_sensor', {}).items():
        sensors_data.append({
            'Sensor': sn,
            'Files': data.get('count', 0),
            'LP AUC': f"{data.get('linear_probe_auc_mean', 0.0):.4f} ± {data.get('linear_probe_auc_std', 0.0):.4f}",
            'LP AP': f"{data.get('linear_probe_ap_mean', 0.0):.4f}",
            'Maha AUC': f"{data.get('mahalanobis_auc_mean', 0.0):.4f}",
            'CNR': f"{data.get('cnr_mean', 0.0):.2f}",
            'Seg IoU': f"{data.get('segmentation_iou_mean', 0.0):.4f}"
        })
    df_sensors = pd.DataFrame(sensors_data)
    display(HTML('<h3>2. Hiệu năng theo Loại Cảm biến (Sensor)</h3>'))
    display(df_sensors)

    # 3. Bảng Canonical Test-Time Adaptation (TTA) Benchmark
    tta_res = summary.get('canonical_test_time_adaptation', {})
    if tta_res and 'shifts' in tta_res:
        tta_data = []
        for shift_axis, data in tta_res['shifts'].items():
            base = data.get('baseline_zero_shot', {})
            tta_lp = data.get('test_time_adapted_lp', {})
            tta_mh = data.get('test_time_adapted_maha', {})
            delta_lp = data.get('deltas', {}).get('delta_lp_auc', 0.0)
            delta_mh = data.get('deltas', {}).get('delta_maha_auc', 0.0)
            tta_data.append({
                'Shift Axis': shift_axis.upper(),
                'Target Files': data.get('target_count', 0),
                'Zero-Shot LP AUC': f"{base.get('mean_auc', 0.0):.4f}",
                'TTA LP AUC': f"{tta_lp.get('mean_auc', 0.0):.4f}",
                'Δ LP AUC': f"{delta_lp:+.4f}",
                'TTA Maha AUC': f"{tta_mh.get('mean_auc', 0.0):.4f}",
                'Δ Maha AUC': f"{delta_mh:+.4f}",
                'TTA LP AP': f"{tta_lp.get('mean_ap', 0.0):.4f}"
            })
        df_tta = pd.DataFrame(tta_data)
        display(HTML('<h3>3. Kết quả Canonical Test-Time Adaptation (TTA) Benchmark</h3>'))
        display(df_tta)
else:
    print(f'[!] Chưa tìm thấy {summary_path}')


In [ ]:
# ==============================================================================
# HIỂN THỊ TRỰC QUAN C-SCAN ANOMALY DETECTION & TTA MAPS
# ==============================================================================
import glob
from PIL import Image
import matplotlib.pyplot as plt

# 1. Hiển thị Anomaly Detection (4-panel subplots: CAD GT, Raw Vpp, Mahalanobis, Linear Probe)
anom_plots = sorted(glob.glob(os.path.join(eval_out_dir, 'task_1_anomaly_detection', '**', '*.png'), recursive=True))
print(f'[*] Tìm thấy {len(anom_plots)} biểu đồ Anomaly Detection.')

if anom_plots:
    for p in anom_plots[:3]:
        img = Image.open(p)
        plt.figure(figsize=(18, 5))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f'Anomaly Detection 4-Panel Map: {os.path.basename(p)}', fontsize=12, fontweight='bold')
        plt.show()

# 2. Hiển thị TTA Comparison (5-panel subplots: CAD GT, Raw Vpp, Zero-Shot LP, TTA LP, TTA Maha)
tta_plots = sorted(glob.glob(os.path.join(eval_out_dir, 'task_4_canonical_tta', '*.png')))
print(f'[*] Tìm thấy {len(tta_plots)} biểu đồ TTA Benchmark.')

if tta_plots:
    for p in tta_plots[:3]:
        img = Image.open(p)
        plt.figure(figsize=(22, 5))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f'TTA Adaptation 5-Panel Map: {os.path.basename(p)}', fontsize=12, fontweight='bold')
        plt.show()

# 3. Hiển thị Sizing Calibration Scatter Plot
sizing_plots = sorted(glob.glob(os.path.join(eval_out_dir, 'task_2_sizing_hurdle', '**', '*.png'), recursive=True))
if sizing_plots:
    img = Image.open(sizing_plots[0])
    plt.figure(figsize=(8, 7))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Sizing Hurdle Calibration Scatter: {os.path.basename(sizing_plots[0])}', fontsize=12, fontweight='bold')
    plt.show()


In [ ]:
# ==============================================================================
# KIỂM TRA ĐỒNG BỘ TOÀN VẸN TRÊN GOOGLE DRIVE
# ==============================================================================
print('=' * 80)
print(f'Kiểm tra thư mục kết quả: {run_dir}')
for root, dirs, files in os.walk(run_dir):
    level = root.replace(run_dir, '').count(os.sep)
    indent = ' ' * 4 * level
    print(f'{indent}{os.path.basename(root)}/ ({len(files)} files)')
    if level <= 2:
        for f in files[:5]:
            fpath = os.path.join(root, f)
            sz = os.path.getsize(fpath) / (1024 * 1024)
            print(f'{indent}    - {f} ({sz:.2f} MB)')
        if len(files) > 5:
            print(f'{indent}    ... và {len(files) - 5} files khác')
print('=' * 80)
print('[OK] Toàn bộ dữ liệu huấn luyện, weights và kết quả đánh giá đã được đồng bộ an toàn trên Google Drive!')
